In [1]:
import pandas as pd
import numpy as np

In [2]:
cleaned_df = pd.read_csv("../data/ev_population_cleaned.csv")
cleaned_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 581 entries, 0 to 580
Data columns (total 9 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   postal_code         581 non-null    float64
 1   ev_count            581 non-null    int64  
 2   avg_electric_range  581 non-null    float64
 3   bev_ratio           581 non-null    float64
 4   avg_base_msrp       581 non-null    float64
 5   lat                 581 non-null    float64
 6   lon                 581 non-null    float64
 7   demand_level        581 non-null    object 
 8   log_ev_count        581 non-null    float64
dtypes: float64(7), int64(1), object(1)
memory usage: 41.0+ KB


In [3]:
cleaned_df.head()

,postal_code,ev_count,avg_electric_range,bev_ratio,avg_base_msrp,lat,lon,demand_level,log_ev_count
0,83854.0,2,-1.414878,1.316834,-0.193959,47.71189,-116.94810,Low,1.098612
1,83858.0,1,-0.626364,-3.750638,-0.193959,47.81137,-116.89644,Low,0.693147
2,83876.0,1,-1.414878,1.316834,-0.193959,47.40077,-116.91895,Low,0.693147
3,97006.0,1,-1.414878,1.316834,-0.193959,45.53043,-122.88429,Low,0.693147
4,97027.0,1,-0.990294,-3.750638,11.940784,45.38059,-122.59479,Low,0.693147


## Feature/Target Split

In [4]:
X = cleaned_df.drop(columns=['demand_level', 'ev_count', 'log_ev_count', 'postal_code'], errors='ignore')
y = cleaned_df['demand_level']

## Train-Test Split

In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## Modeling

### Logistic Regression

In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.pipeline import Pipeline

# Create and train model
clf = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
clf.fit(X_train, y_train)

# Predictions
y_pred_logreg = clf.predict(X_test)

# Evaluation
print("Accuracy:", round(accuracy_score(y_test, y_pred_logreg), 4))
print("\nClassification Report:\n", classification_report(y_test, y_pred_logreg))

Accuracy: 0.5043

Classification Report:
               precision    recall  f1-score   support

        High       0.56      0.87      0.68        46
         Low       0.42      0.53      0.47        32
      Medium       0.33      0.05      0.09        39

    accuracy                           0.50       117
   macro avg       0.44      0.48      0.41       117
weighted avg       0.45      0.50      0.43       117



### K Nearest Neighbors

In [7]:
from sklearn.neighbors import KNeighborsClassifier

knc = KNeighborsClassifier(n_neighbors=3, weights='distance')
knc.fit(X_train, y_train)

# Predictions
y_pred_knc = knc.predict(X_test)

# Evaluation
print("KNN Accuracy:", round(accuracy_score(y_test, y_pred_knc), 4))
print("\nKNN Classification Report:\n", classification_report(y_test, y_pred_knc))

KNN Accuracy: 0.7265

KNN Classification Report:
               precision    recall  f1-score   support

        High       0.73      0.80      0.76        46
         Low       0.87      0.81      0.84        32
      Medium       0.61      0.56      0.59        39

    accuracy                           0.73       117
   macro avg       0.73      0.73      0.73       117
weighted avg       0.73      0.73      0.72       117



### Decision Tree Classifier

In [8]:
from sklearn.tree import DecisionTreeClassifier

dtc = DecisionTreeClassifier(random_state=42, class_weight='balanced')
dtc.fit(X_train, y_train)

# Predictions
y_pred_dtc = dtc.predict(X_test)

# Evaluation
print("Decision Tree Accuracy:", round(accuracy_score(y_test, y_pred_dtc), 4))
print("\nDecision Tree Classification Report:\n", classification_report(y_test, y_pred_dtc))

Decision Tree Accuracy: 0.7094

Decision Tree Classification Report:
               precision    recall  f1-score   support

        High       0.79      0.72      0.75        46
         Low       0.79      0.84      0.82        32
      Medium       0.56      0.59      0.57        39

    accuracy                           0.71       117
   macro avg       0.71      0.72      0.71       117
weighted avg       0.71      0.71      0.71       117



### Random Forest Classifier

In [9]:
from sklearn.ensemble import RandomForestClassifier

rfc = RandomForestClassifier(random_state=42, class_weight='balanced', n_estimators=100)
rfc.fit(X_train, y_train)

# Predictions
y_pred_rfc = rfc.predict(X_test)

# Evaluation
print("Random Forest Accuracy:", round(accuracy_score(y_test, y_pred_rfc), 4))
print("\nRandom Forest Classification Report:\n", classification_report(y_test, y_pred_rfc))

Random Forest Accuracy: 0.7863

Random Forest Classification Report:
               precision    recall  f1-score   support

        High       0.85      0.72      0.78        46
         Low       0.91      0.91      0.91        32
      Medium       0.65      0.77      0.71        39

    accuracy                           0.79       117
   macro avg       0.80      0.80      0.80       117
weighted avg       0.80      0.79      0.79       117



### XGBoost

In [11]:
from xgboost import XGBClassifier

xgb = XGBClassifier(use_label_encoder=False, eval_metric='mlogloss')

In [12]:
from sklearn.preprocessing import LabelEncoder

# Create encoder and apply it to y_train and y_test
le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_test_encoded = le.transform(y_test)

xgb.fit(X_train, y_train_encoded)
y_pred_xgb = xgb.predict(X_test)
y_pred_xgb_labels = le.inverse_transform(y_pred_xgb)

print("XGBoost Accuracy:", round(accuracy_score(y_test, y_pred_xgb_labels), 4))
print("\nXGBoost Classification Report:\n", classification_report(y_test, y_pred_xgb_labels))

/Users/sowmyamaddali/Desktop/Sowmya/Git/ev-market-size-case-study/ev_env/lib/python3.11/site-packages/xgboost/training.py:183: UserWarning: [15:22:50] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


XGBoost Accuracy: 0.7692

XGBoost Classification Report:
               precision    recall  f1-score   support

        High       0.77      0.78      0.77        46
         Low       0.93      0.88      0.90        32
      Medium       0.65      0.67      0.66        39

    accuracy                           0.77       117
   macro avg       0.78      0.77      0.78       117
weighted avg       0.77      0.77      0.77       117



## Compare Model Performance

In [14]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

results_classification = pd.DataFrame({
    'Model': ['Logistic Regression', 'KNearest Neighbors', 'Decision Tree Classifier', 'Random Forest', 'XGBoost'],
    'Accuracy': [
        round(accuracy_score(y_test, y_pred_logreg), 4),
        round(accuracy_score(y_test, y_pred_knc), 4),
        round(accuracy_score(y_test, y_pred_dtc), 4),
        round(accuracy_score(y_test, y_pred_rfc), 4),
        round(accuracy_score(y_test, y_pred_xgb_labels), 4)
    ],
    'Precision (Macro)': [
        round(precision_score(y_test, y_pred_logreg, average='macro'), 4),
        round(precision_score(y_test, y_pred_knc, average='macro'), 4),
        round(precision_score(y_test, y_pred_dtc, average='macro'), 4),
        round(precision_score(y_test, y_pred_rfc, average='macro'), 4),
        round(precision_score(y_test, y_pred_xgb_labels, average='macro'), 4)
    ],
    'Recall (Macro)': [
        round(recall_score(y_test, y_pred_logreg, average='macro'), 4),
        round(recall_score(y_test, y_pred_knc, average='macro'), 4),
        round(recall_score(y_test, y_pred_dtc, average='macro'), 4),
        round(recall_score(y_test, y_pred_rfc, average='macro'), 4),
        round(recall_score(y_test, y_pred_xgb_labels, average='macro'), 4)
    ],
    'F1 Score (Macro)': [
        round(f1_score(y_test, y_pred_logreg, average='macro'), 4),
        round(f1_score(y_test, y_pred_knc, average='macro'), 4),
        round(f1_score(y_test, y_pred_dtc, average='macro'), 4),
        round(f1_score(y_test, y_pred_rfc, average='macro'), 4),
        round(f1_score(y_test, y_pred_xgb_labels, average='macro'), 4)
    ]
})

print(results_classification)

                      Model  Accuracy  Precision (Macro)  Recall (Macro)  \
0       Logistic Regression    0.5043             0.4406          0.4840   
1        KNearest Neighbors    0.7265             0.7344          0.7270   
2  Decision Tree Classifier    0.7094             0.7136          0.7170   
3             Random Forest    0.7863             0.8015          0.7976   
4                   XGBoost    0.7692             0.7831          0.7748   

   F1 Score (Macro)  
0            0.4150  
1            0.7294  
2            0.7144  
3            0.7962  
4            0.7785  


In [15]:
# Assuming your target column is named 'demand_level'
print(cleaned_df['demand_level'].value_counts())

demand_level
Low       194
High      194
Medium    193
Name: count, dtype: int64


## Fine-Tuning Random Forest

In [18]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

# Define the model
rf = RandomForestClassifier(random_state=42)

# Define the hyperparameter grid
param_grid_refined = {
    'n_estimators': [250, 300, 350],
    'max_depth': [8, 10, 12],
    'min_samples_split': [2, 3, 5],
    'min_samples_leaf': [1, 2, 3]
}

# # Set up GridSearchCV
# grid_search = GridSearchCV(estimator=rf,
#                            param_grid=param_grid,
#                            cv=5,
#                            n_jobs=-1,
#                            scoring='f1_macro',
#                            verbose=2)

refined_grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=param_grid_refined,
    cv=5,
    n_jobs=-1,
    scoring='f1_macro',
    verbose=2
)

# Fit the model
grid_search.fit(X_train, y_train)

# Best model after tuning
best_rf = grid_search.best_estimator_

# Predict and evaluate
y_pred_best_rf = best_rf.predict(X_test)

Fitting 5 folds for each of 108 candidates, totalling 540 fits
[CV] END max_depth=None, min_samples_leaf=1, min_samples_split=2, n_estimators=100; total time=   0.1s
[CV] END max_depth=None, min_samples_leaf=1, min_samples_split=2, n_estimators=100; total time=   0.1s
[CV] END max_depth=None, min_samples_leaf=1, min_samples_split=2, n_estimators=100; total time=   0.1s
[CV] END max_depth=None, min_samples_leaf=1, min_samples_split=2, n_estimators=100; total time=   0.1s
[CV] END max_depth=None, min_samples_leaf=1, min_samples_split=2, n_estimators=100; total time=   0.1s
[CV] END max_depth=None, min_samples_leaf=1, min_samples_split=2, n_estimators=200; total time=   0.2s
[CV] END max_depth=None, min_samples_leaf=1, min_samples_split=2, n_estimators=200; total time=   0.2s
[CV] END max_depth=None, min_samples_leaf=1, min_samples_split=2, n_estimators=200; total time=   0.2s
[CV] END max_depth=None, min_samples_leaf=1, min_samples_split=2, n_estimators=200; total time=   0.2s
[CV] END m

In [ ]:
from sklearn.metrics import classification_report, accuracy_score

print("Best Parameters:\n", grid_search.best_params_)
print("\nAccuracy:", round(accuracy_score(y_test, y_pred_best_rf), 4))
print("\nClassification Report:\n", classification_report(y_test, y_pred_best_rf))

Best Parameters:
 {'max_depth': 10, 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 300}

Accuracy: 0.7949

Classification Report:
               precision    recall  f1-score   support

        High       0.85      0.74      0.79        46
         Low       0.91      0.91      0.91        32
      Medium       0.67      0.77      0.71        39

    accuracy                           0.79       117
   macro avg       0.81      0.80      0.80       117
weighted avg       0.80      0.79      0.80       117



## Saving the model

In [ ]:
import joblib

# Save the model
joblib.dump(best_rf, '../models/random_forest_classifier.pkl')